In [92]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-4
eval_iters = 250

cuda


In [ ]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [122]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)

In [124]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')

In [126]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [128]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


6,!UyVq:rY(Wl8XIHP[tP3﻿jmsf18P3kYCoHhOM"
F6L"h_Qm97C?bR0E?j1[F],lQRZkYJFg6*W7h3k69[]1g4jV:xiXogZjRD*td6ta_5nJzDjd,E3Mhg﻿khVgRDSucVt2;Jzl*WK Z'*r[F91﻿QRlAK-9n8*Y8g,b]x.5lWFwCAZ5HM
O0i6POR0p4[JmgXr,gX!0iQa5H!f1.dpJ3OhPO0V]1FbDYN,XG*rC(﻿A6*v3TDt]WAZ88:.Vw1-P8ar"*Z[) ZSV.vWlWr!"GJ-2oWd_T6Q Vn8:4]bDSK0Qtfdm4Y!m(2AA﻿mH?
Waou-ZS[tEL3ScOT?EVPGdq:GJKpXQ0LrC?U3Jal]2U,u'*(7.V3KMON57dS(p0JKb"?fU6]﻿.YdP&VhAmZ.n6)G2A2dDSGJYZ.Rg&cT(T(rt4zHel)55"nrXt?gkJbA(Eyq3,(?j2A
px4dI3U9"smpHF*4q057g.!4C]mLVqxwqRkSVUulcYB﻿


In [130]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']}, val loss: {losses['val']}")
        
    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step: 0, train loss: 4.961136341094971, val loss: 4.925435543060303
step: 250, train loss: 4.881243705749512, val loss: 4.891665458679199
step: 500, train loss: 4.837502479553223, val loss: 4.824849605560303
step: 750, train loss: 4.748220920562744, val loss: 4.755435466766357
4.692161560058594


In [131]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


ciU1L&rzT("T(&rGBF]xU9[]4'8O]R.wrT(v
1&CMA;oa4,uRkYJKAusg-ajII
8r]FYAQY1j]60KP"D[o]RjuNfpEMdD[MdHFYp_;nI02GJs-k.0
Z8d7&J9ltIj6qn5A
﻿BsGat
]b0:'z0i5lv2U9T(vjgdX?joaltaM1lEQU1f?E2g,d1pi(oYB6k LF0F9tnfn:OH2A!AMscYkng 8:;Nly:LIO][F)s18"Th;.!2i50p
qeA(oApY1F&rzr HU1NQv16qA4VPHAmD]Jz,Xb,13.0JuYyUCN(VUR9yalyGJaVvn8xiv.oaly0dD'nvb5EcTcY!h1o7K﻿QbgG3T(2*yYqa﻿B!W,)OmFtrH_u8C2(oJsCjIM! 32li6nQFx1MQRP"sjyG5vFcT(T([ Y!v1.W1lk.!2&bZ[cc3PmY9Fxnyd_]6s&r7G
D(pa]R8FBFbRmKbn5GJBmA4DXbG7dwYBNXtFxpp97*GJHA6O E]f_:n0u
